# Photometry / Astrometry Pipeline — Full Walkthrough

This notebook is a hands-on companion to the reference docs in `docs/`. It walks through every stage of the pipeline using the real functions from `steps_photometry.py`, `steps_astrometry.py`, and `steps_zeropoint.py` — the same code used by `photometry_pipeline.py`.

**Audience:** observatory staff who will run this pipeline but did not build it.

**A note on interactivity:** the real pipeline asks you questions at several points (which astrometry method to use, whether to accept a set of candidate stars, RA/Dec entry, etc.). Those steps use Python's `input()`, which does not run well inside a notebook cell. Where that happens below, we call the same underlying functions with example values instead, and note clearly that *"in the real pipeline, this step will prompt you interactively."*

**Stages covered in this notebook:**
1. Overview
2. Star detection & FWHM measurement
3. Astrometry (all three tiers)
4. Zero-point calibration
5. Saving results
6. Running the full pipeline end-to-end

> Run this notebook from the same folder as `photometry_pipeline.py`, `steps_photometry.py`, `steps_astrometry.py`, `steps_zeropoint.py`, and `config.py`, so the imports below work without changes.

---
## 1. Overview

The pipeline takes a folder of calibrated FITS images and produces, for each image, a table of real, catalog-calibrated star magnitudes.

```
FITS images
    |
    v
Detected sources (x, y, flux)      <- steps_photometry.find_stars
    |
    v
Measured FWHM                       <- steps_photometry.estimate_fwhm_ensemble
    |
    v
WCS (sky pointing solution)         <- steps_astrometry (3 possible methods)
    |
    v
RA / Dec per source                 <- steps_photometry.build_photometry_table
    |
    v
Pan-STARRS catalog match            <- steps_zeropoint.match_sources_to_catalog
    |
    v
Zero-point (mag offset)             <- steps_zeropoint.compute_zeropoint
    |
    v
Real, calibrated magnitudes         <- steps_zeropoint.apply_zeropoint
    |
    v
Saved FITS table per image          <- steps_photometry.save_photometry_table
```

Before running anything, make sure `config.py` has your real astrometry.net API key and the correct scale hint (`SCALE_LOWER_ARCMIN`, `SCALE_UPPER_ARCMIN`) for your current optical setup — see `docs/00_overview.md` for details.

In [ ]:
# Imports used throughout this notebook
import os
import numpy as np
from astropy.io import fits
from astropy.table import Table
from astropy import conf

from config import ASTROMETRY_API_KEY, SCALE_LOWER_ARCMIN, SCALE_UPPER_ARCMIN

from steps_photometry import (
    find_stars,
    estimate_fwhm_ensemble,
    build_photometry_table,
    save_photometry_table,
    create_dict_aper,
)
from steps_astrometry import (
    select_astrometric_candidates,
    show_astrometric_candidates,
    review_astrometric_candidates,
    solve_with_astrometry_net,
    run_manual_astrometry,
    get_position_hint_from_user,
)
from steps_zeropoint import (
    query_field_catalog,
    match_sources_to_catalog,
    compute_zeropoint,
    apply_zeropoint,
)

print("Imports OK")

In [ ]:
# EDIT THIS: path to one real calibrated FITS image from your data
IMAGE_PATH = r"YOUR PATH"

with fits.open(IMAGE_PATH) as hdul:
    image_data = hdul[0].data.astype(float)

print(f"Loaded {os.path.basename(IMAGE_PATH)}, shape={image_data.shape}")

---
## 2. Star Detection & FWHM Measurement

See `docs/01_detection_and_fwhm.md` for the full parameter reference.

`find_stars` detects stars using DAOStarFinder, with an optional cleanup pass that removes hot pixels and cosmic rays. `estimate_fwhm_ensemble` then measures how sharp the brightest stars are, which later steps use to size apertures and pick reasonable candidate stars.

In [ ]:
found_stars = find_stars(
    image_path=IMAGE_PATH,
    threshold_factor=6.0,   # higher = fewer, brighter-only detections
    apply_filter=True        # removes hot pixels / cosmic rays
)

print(f"Detected {len(found_stars)} sources")
found_stars[:5]  # preview the first 5 detections

In [ ]:
fwhm, n_used = estimate_fwhm_ensemble(found_stars, image_data)
print(f"Measured FWHM: {fwhm:.2f} px, from {n_used} stars")

# Known gotcha: if found_stars were empty (e.g. a dark frame with no real
# stars), estimate_fwhm_ensemble currently returns a bare None instead of
# a (None, 0) tuple. Always sanity-check your input folder only contains
# real science frames, not calibration frames (darks/bias/flats).

---
## 3. Astrometry (Sky Coordinate Solving)

See `docs/02_astrometry.md` for the full reference. Each image is solved **independently** — dithered exposures point at slightly different sky positions, so there is no shared WCS across a batch.

Solving happens in up to three tiers, only escalating if the previous tier fails on **every image** in a batch:

1. **Automatic, blind** — astrometry.net, no position hint
2. **Automatic, guided** — astrometry.net, with a position hint from one manually-identified star
3. **Fully manual** — fit a WCS from several manually identified stars (reference image only)

### 3a. Tier 1 — automatic, blind

In [ ]:
wcs = solve_with_astrometry_net(
    sources=found_stars,
    image_shape=image_data.shape,
    api_key=ASTROMETRY_API_KEY,
    scale_lower_arcmin=SCALE_LOWER_ARCMIN,
    scale_upper_arcmin=SCALE_UPPER_ARCMIN
)

if wcs is not None:
    print("Solved successfully.")
    sky = wcs.pixel_to_world(found_stars["xcentroid"], found_stars["ycentroid"])
    print(f"Field center: RA={np.mean(sky.ra.deg):.5f}, Dec={np.mean(sky.dec.deg):.5f}")
else:
    print("Solve failed — see the printed suggestions above for likely causes.")

### 3b. Tier 2 — guided retry with a position hint (demo only)

**In the real pipeline**, this is only offered if Tier 1 fails on *every* image in a batch. It shows a numbered plot of candidate stars and asks which one you can identify, then asks for its known RA/Dec (`HH:MM:SS.ss` / `+/-DD:MM:SS.s`).

Below, we call the same function directly with an example star identification, to show the mechanics without needing a live prompt. Skip this cell if Tier 1 already succeeded above.

In [ ]:
candidates = select_astrometric_candidates(
    sources=found_stars,
    image_shape=image_data.shape,
    fwhm=fwhm
)

show_astrometric_candidates(image_data, candidates, title="Tier 2 demo: candidate stars")

# In the real pipeline: get_position_hint_from_user(image_data, candidates)
# prompts interactively for a star number, then its RA/Dec.
# Example values shown here purely to demonstrate the search-radius calculation:

example_ra_hint = 359.552   # degrees, e.g. from get_position_hint_from_user(...)
example_dec_hint = 61.212   # degrees

# The search radius scales automatically with the current optical setup,
# rather than a fixed arcmin number, so it stays correct if the FOV changes
# later (e.g. a focal reducer is added):
search_radius_arcmin = SCALE_UPPER_ARCMIN * 4
print(f"Would retry with RA={example_ra_hint}, Dec={example_dec_hint}, "
      f"search_radius_arcmin={search_radius_arcmin}")

# wcs_retry = solve_with_astrometry_net(
#     sources=found_stars, image_shape=image_data.shape, api_key=ASTROMETRY_API_KEY,
#     scale_lower_arcmin=SCALE_LOWER_ARCMIN, scale_upper_arcmin=SCALE_UPPER_ARCMIN,
#     ra=example_ra_hint, dec=example_dec_hint, search_radius_arcmin=search_radius_arcmin
# )

### 3c. Tier 3 — fully manual astrometry (demo only)

**In the real pipeline**, this is the last resort if Tier 2 also fails on every image. `review_astrometric_candidates` lets you reject fake detections (hot pixels, artifacts) from the candidate list; `run_manual_astrometry` then asks for RA/Dec for each remaining star, fits a WCS, and checks the fit quality (residual between what you typed and what the fitted WCS predicts back).

This cell is illustrative — running it for real requires typing real RA/Dec values at each prompt.

In [ ]:
# Uncomment to run interactively (requires real RA/Dec input at each prompt):

# reviewed_candidates = review_astrometric_candidates(found_stars, image_data, fwhm)
# manual_wcs = run_manual_astrometry(image_data, reviewed_candidates, min_stars=3)
#
# run_manual_astrometry requires at least 3 identified stars, and after fitting
# checks the RMS residual between your typed RA/Dec and what the fitted WCS
# predicts back from the same pixel positions. If the residual exceeds
# warn_threshold_arcsec (default 1.0 arcsec), you're warned and offered a
# chance to redo entry rather than silently accepting a bad fit.

print("See docs/02_astrometry.md for the full manual-entry walkthrough.")

---
## 4. Zero-Point Calibration

See `docs/03_zeropoint_calibration.md` for the full reference.

Raw aperture photometry gives an *instrumental* magnitude with an arbitrary zero point. To get a real, standard magnitude, we compare our detections against Pan-STARRS DR2:

```
real_mag = inst_mag + zero_point
```

The catalog is queried **once per image field**, not once per star — this is both faster and scales correctly whether the field is sparse or densely packed with stars.

In [ ]:
# Build a minimal photometry table for this demo (normally produced by
# build_photometry_table as part of the full pipeline run — see Section 6)

if wcs is not None:
    sky = wcs.pixel_to_world(found_stars["xcentroid"], found_stars["ycentroid"])

    phot_table = Table()
    phot_table["x"] = found_stars["xcentroid"]
    phot_table["y"] = found_stars["ycentroid"]
    phot_table["RA"] = sky.ra.deg
    phot_table["Dec"] = sky.dec.deg
    phot_table["inst_mag"] = -2.5 * np.log10(found_stars["flux"])

    ra_center = float(np.mean(phot_table["RA"]))
    dec_center = float(np.mean(phot_table["Dec"]))

    catalog = query_field_catalog(ra_center, dec_center, radius_arcmin=2.0, ps1_filter='g')
    print(f"Pan-STARRS returned {len(catalog) if catalog is not None else 0} catalog sources")
else:
    print("No WCS available from Tier 1 above — skipping zero-point demo. "
          "Re-run Section 3a, or use a dataset that solves successfully.")

In [ ]:
if wcs is not None and catalog is not None:
    matched = match_sources_to_catalog(phot_table, catalog, ps1_filter='g', max_sep_arcsec=1.0)
    print(f"Matched {len(matched) if matched is not None else 0} sources against Pan-STARRS")

    zp, zp_sigma, n_used = compute_zeropoint(matched)

    phot_table = apply_zeropoint(phot_table, zp)
    print("\nFull table with calibrated magnitudes:")
    conf.max_lines = -1
    conf.max_width = -1
    print(phot_table)

**Sanity check:** since dithered exposures of the same field should produce nearly identical zero-points, comparing `zp` across a full batch of images is a good way to catch upstream problems — wildly scattered zero-points usually mean bad astrometry or contaminated detections somewhere, not a calibration bug.

---
## 5. Saving Results

See `docs/04_saving_results.md` for the full reference.

Each image's final table is saved as a FITS binary table, following the same convention used by professional ground-based pipelines (ESO, WFST, ZTF): the zero-point and data provenance are stored directly in the FITS header, not just applied and discarded.

In [ ]:
if wcs is not None and catalog is not None:
    out_path = save_photometry_table(
        photometry_table=phot_table,
        output_dir=os.path.dirname(IMAGE_PATH),
        original_filename=os.path.basename(IMAGE_PATH),
        original_path=IMAGE_PATH,
        zp=zp,
        zp_sigma=zp_sigma,
        n_zp_stars=n_used,
        fwhm=fwhm,
        aperture_radius=1.5 * fwhm,
        astrometry_method="astrometry_net",
        photsys="PS1-g"
    )
    print(f"Saved to: {out_path}")

### Reading a saved result back

A saved `.fits` result cannot be meaningfully opened in Notepad (the table data is binary) or properly browsed in DS9 (built for images, not tables). Read it back with `astropy` instead:

In [ ]:
if wcs is not None and catalog is not None:
    conf.max_lines = -1
    conf.max_width = -1

    t = Table.read(out_path)
    print(t)

    with fits.open(out_path) as hdul:
        print(f"\nZero-point stored in header: MAGZP={hdul[1].header['MAGZP']}, "
              f"MAGZPERR={hdul[1].header['MAGZPERR']}")

For casual browsing without writing code, [TOPCAT](https://www.star.bris.ac.uk/~mbt/topcat/) opens FITS binary tables as a spreadsheet-like view.

---
## 6. Running the Full Pipeline End-to-End

See `docs/05_running_the_full_pipeline.md` for the complete prompt-by-prompt reference and troubleshooting guide.

In practice, you will not call the individual functions above by hand — you run the orchestration script directly from a terminal:

```
python photometry_pipeline.py
```

It will ask you, in order:

1. **Image folder** — path to your calibrated science FITS images (not darks/bias/flats)
2. **Detection settings** — `n` for defaults (recommended), `y` to fine-tune
3. **Astrometry method** — `1` for automatic (recommended), `2` for manual
4. *(automatic)* detection, FWHM, and astrometric solving for every image
5. *(only if every image fails to solve)* a guided position-hint retry, then a fully manual fallback
6. *(automatic)* Pan-STARRS zero-point calibration for every solved image
7. **Results** saved to `<your_folder>/photometry_results/` and printed to the terminal

This notebook demonstrated each of those stages individually, on one image, so you can see exactly what the pipeline is doing under the hood — but for routine use, just run the script above and follow the prompts.